# Corporate tax rates by country: the Middle East first, then the world

**What rate of tax does a company pay on its profits in each Middle East economy, and how does the region compare with the rest of the world?**

Data: Tax Foundation, *Corporate Tax Rates around the World 2025* (226 jurisdictions), checked against the PwC Worldwide Tax Summaries quick chart and the Trading Economics country list, both read in October 2026.

This notebook loads the raw readings, checks them, recomputes the headline figures and asserts that they match the output of `scripts/build_data.py`.

## 1. Load the raw files

The Tax Foundation table was read twice, in two separate passes. Each pass is kept as its own file.

In [1]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # run from the project folder
import pandas as pd
pd.set_option("display.width", 160)
a = pd.read_csv("data/raw/tax_foundation_2025_reading_a.csv")
b = pd.read_csv("data/raw/tax_foundation_2025_reading_b.csv")
pwc = pd.read_csv("data/raw/pwc_quick_chart_2026.csv")
te = pd.read_csv("data/raw/trading_economics_2026.csv")
print("Tax Foundation reading A:", a.shape, " reading B:", b.shape)
print("PwC quick chart:", pwc.shape, " Trading Economics:", te.shape)
print(a.head().to_string(index=False))

Tax Foundation reading A: (226, 4)  reading B: (226, 3)
PwC quick chart: (146, 3)  Trading Economics: (162, 3)
iso3       country  rate_pct  rate_with_gmt_pct
 ABW         Aruba      22.0               22.0
 AFG   Afghanistan      20.0               20.0
 AGO        Angola      25.0               25.0
 AIA      Anguilla       0.0                0.0
 ALA Aland Islands      20.0               20.0


## 2. Data checks

Row count, missing values, duplicates, value range, and whether the two readings of the table agree.

In [2]:
print("missing rates:", int(a.rate_pct.isna().sum()), "| missing rate with minimum tax:", int(a.rate_with_gmt_pct.isna().sum()),
      "(" + ", ".join(a[a.rate_with_gmt_pct.isna()].country) + ")")
print("duplicate codes:", int(a.iso3.duplicated().sum()), int(b.iso3.duplicated().sum()))
print("rate range:", a.rate_pct.min(), "to", a.rate_pct.max())
m = a.merge(b, on="iso3", suffixes=("_a", "_b"))
print("jurisdictions in both readings:", len(m))
print("rates that differ between the two readings:", int((m.rate_pct_a != m.rate_pct_b).sum()))
assert len(m) == 226 and (m.rate_pct_a == m.rate_pct_b).all()

missing rates: 0 | missing rate with minimum tax: 1 (Poland)
duplicate codes: 0 0
rate range: 0.0 to 50.0
jurisdictions in both readings: 226
rates that differ between the two readings: 0


## 3. Do the totals printed by the source match the table?

The Tax Foundation text prints the average of all 226 rates and the number of jurisdictions in several rate bands. A misread row would move at least one of them.

In [3]:
r = a.rate_pct
print("mean of 226 rates:", round(r.mean(), 2), "(published 22.35)")
print("no corporate income tax:", int((r == 0).sum()), "(published 15)")
print("at or below 20%:", int((r <= 20).sum()), "(published 80)")
print("above 20% and at or below 30%:", int(((r > 20) & (r <= 30)).sum()), "(published 126)")
print("above 30% and at or below 35%:", int(((r > 30) & (r <= 35)).sum()), "(published 16)")
print("above 35%:", int((r > 35).sum()), "(published 4)")
assert round(r.mean(), 2) == 22.35 and (r == 0).sum() == 15 and (r <= 20).sum() == 80
assert ((r > 20) & (r <= 30)).sum() == 126 and ((r > 30) & (r <= 35)).sum() == 16 and (r > 35).sum() == 4

mean of 226 rates: 22.35 (published 22.35)
no corporate income tax: 15 (published 15)
at or below 20%: 80 (published 80)
above 20% and at or below 30%: 126 (published 126)
above 30% and at or below 35%: 16 (published 16)
above 35%: 4 (published 4)


## 4. Build the analysis table

The build script adds regions, matches the PwC and Trading Economics listings by country, and labels each rate `confirmed`, `differs` or `single-source`.

In [4]:
import sys
sys.path.insert(0, "scripts")
import build_data
df, reg, pub = build_data.build()
saved = pd.read_csv("data/processed/corporate_tax_tidy.csv")
assert len(df) == len(saved) == 226
assert (df.rate_2025_pct.values == saved.rate_2025_pct.values).all() and (df.iso3.values == saved.iso3.values).all()
print(df.check_status.value_counts().to_string())
print()
print(pub[["check", "published_value", "computed_value", "result"]].to_string(index=False))

check_status
confirmed        150
single-source     54
differs           22

                                check  published_value  computed_value                          result
               jurisdictions_in_table           226.00          226.00                           match
      mean_rate_all_jurisdictions_pct            22.35           22.35                           match
jurisdictions_no_corporate_income_tax            15.00           15.00                           match
     jurisdictions_at_or_below_20_pct            80.00           80.00                           match
     jurisdictions_above_20_to_30_pct           126.00          126.00                           match
     jurisdictions_at_or_below_30_pct           206.00          206.00                           match
     jurisdictions_above_30_to_35_pct            16.00           16.00                           match
           jurisdictions_above_35_pct             4.00            4.00                           ma

## 5. The Middle East, economy by economy

In [5]:
me = df[df.region == "Middle East"]
order = ["Gulf", "Levant and Iraq", "Iran, Turkey and Yemen", "North Africa"]
me = me.assign(o=me.sub_region.map({s: i for i, s in enumerate(order)})).sort_values(["o", "rate_2025_pct", "country"])
print(me[["sub_region", "country", "rate_2025_pct", "rate_with_min_tax_2025_pct", "pwc_rate_2026_pct",
          "te_rate_2026_pct", "check_status"]].to_string(index=False))

            sub_region              country  rate_2025_pct  rate_with_min_tax_2025_pct  pwc_rate_2026_pct  te_rate_2026_pct  check_status
                  Gulf              Bahrain            0.0                        15.0                0.0               0.0     confirmed
                  Gulf United Arab Emirates            9.0                        15.0                9.0               9.0     confirmed
                  Gulf                Qatar           10.0                        15.0               10.0              10.0     confirmed
                  Gulf               Kuwait           15.0                        15.0               15.0              15.0     confirmed
                  Gulf                 Oman           15.0                        15.0               15.0              15.0     confirmed
                  Gulf         Saudi Arabia           20.0                        20.0               20.0              20.0     confirmed
       Levant and Iraq            

![Middle East chart](../images/corporate-tax-middle-east-chart.png)

## 6. Headline figures

Each figure is recomputed here from the tidy table and compared with the build output.

In [6]:
world = round(df.rate_2025_pct.mean(), 2)
gulf = df[df.sub_region == "Gulf"]
gulf_mean = round(gulf.rate_2025_pct.mean(), 2)
gulf_mean_min_tax = round(gulf.rate_with_min_tax_2025_pct.mean(), 2)
me_mean = round(me.rate_2025_pct.mean(), 2)
below = int((me.rate_2025_pct < world).sum())
print("World mean, 226 jurisdictions:", world)
print("Middle East mean, 20 economies:", me_mean, "| range", me.rate_2025_pct.min(), "to", me.rate_2025_pct.max())
print("Middle East economies below the world mean:", below, "of", len(me))
print("Gulf mean:", gulf_mean, "| with the 15% minimum tax:", gulf_mean_min_tax)
print("Gulf states at or below 15%:", int((gulf.rate_2025_pct <= 15).sum()), "of", len(gulf))
print("Jurisdictions with a rate above zero and below the UAE's 9%:",
      ", ".join(df[(df.rate_2025_pct > 0) & (df.rate_2025_pct < 9)].country))
row = reg[reg.group == "Middle East"].iloc[0]
assert (world, me_mean, below, gulf_mean, gulf_mean_min_tax) == (22.35, 18.82, 13, 11.5, 15.83)
assert row.mean_rate_pct == me_mean and reg[reg.group == "Gulf"].iloc[0].mean_rate_pct == gulf_mean

World mean, 226 jurisdictions: 22.35
Middle East mean, 20 economies: 18.82 | range 0.0 to 34.0
Middle East economies below the world mean: 13 of 20
Gulf mean: 11.5 | with the 15% minimum tax: 15.83
Gulf states at or below 15%: 5 of 6
Jurisdictions with a rate above zero and below the UAE's 9%: Turkmenistan


## 7. Region by region, Middle East first

In [7]:
print(reg[reg.level != "world"].to_string(index=False))
print()
print(reg[reg.level == "world"].to_string(index=False))

                 level                  group  jurisdictions  mean_rate_pct  median_rate_pct  min_rate_pct  max_rate_pct  at_or_below_15_pct  confirmed
                region            Middle East             20          18.82            20.00           0.0         34.00                   7         16
                region                 Europe             51          18.59            20.00           0.0         36.13                  17         40
                region           Rest of Asia             33          20.87            20.00           8.0         30.00                   6         25
                region         Rest of Africa             50          27.67            27.75          15.0         50.00                   2         32
                region          North America             38          21.52            25.68           0.0         37.50                   9         16
                region          South America             13          28.19            2

![Regions chart](../images/corporate-tax-by-region.png)

## 8. Does the choice of world average change the picture?

The simple mean of all 226 jurisdictions is 22.35%. The Tax Foundation also prints a mean of the 181 jurisdictions that have GDP data (23.58%) and a GDP-weighted mean (26.04%). The comparison is run against all three.

In [8]:
for label, w in [("simple mean, 226 jurisdictions", 22.35), ("simple mean, 181 jurisdictions (published)", 23.58),
                 ("GDP-weighted mean, 181 jurisdictions (published)", 26.04), ("median, 226 jurisdictions", df.rate_2025_pct.median())]:
    print(f"{label}: {w:.2f}% | Middle East economies below it: {int((me.rate_2025_pct < w).sum())} of 20"
          f" | Gulf states below it: {int((gulf.rate_2025_pct < w).sum())} of 6")

simple mean, 226 jurisdictions: 22.35% | Middle East economies below it: 13 of 20 | Gulf states below it: 6 of 6
simple mean, 181 jurisdictions (published): 23.58% | Middle East economies below it: 15 of 20 | Gulf states below it: 6 of 6
GDP-weighted mean, 181 jurisdictions (published): 26.04% | Middle East economies below it: 19 of 20 | Gulf states below it: 6 of 6
median, 226 jurisdictions: 25.00% | Middle East economies below it: 15 of 20 | Gulf states below it: 6 of 6


## 9. Where the 2025 table and the 2026 listings differ

These rates did not match either second listing. Some are 2026 changes (Morocco, Lithuania). Others reflect a different definition: the Tax Foundation figure is a combined rate with surcharges and local taxes, the listings print the central rate.

In [9]:
print(df[df.check_status == "differs"][["country", "region", "rate_2025_pct", "pwc_rate_2026_pct", "te_rate_2026_pct"]].to_string(index=False))
print()
print("Middle East rates resting on the Tax Foundation table alone:",
      ", ".join(me[me.check_status == "single-source"].country))

      country         region  rate_2025_pct  pwc_rate_2026_pct  te_rate_2026_pct
        Syria    Middle East          25.00                NaN             28.00
      Morocco    Middle East          34.00               35.0             35.00
    Lithuania         Europe          16.00               17.0             17.00
  Switzerland         Europe          19.61                NaN             14.40
        Italy         Europe          27.81               24.0             24.00
      Germany         Europe          30.06                NaN             30.00
     Portugal         Europe          30.50               19.0             19.00
       France         Europe          36.13               25.0             25.00
  South Korea   Rest of Asia          26.40               25.0             25.00
        Japan   Rest of Asia          29.74               23.2             31.52
        India   Rest of Asia          30.00                NaN             34.94
   Cabo Verde Rest of Africa

![Gulf comparison](../images/corporate-tax-gulf-comparison.png)